In [ ]:
pip install pandas numpy matplotlib scikit-learn seaborn

In [ ]:
# Импорт библиотек — как подключение инструментов, нужных для работы
import pandas as pd  # Для работы с табличными данными
import numpy as np   # Для математических операций
import matplotlib.pyplot as plt # модуль для построения графиков и визуализаций.
from sklearn.preprocessing import LabelEncoder # переводит категориальные (текстовые) значения в числовые коды.
import seaborn as sns # библиотека статистической визуализации поверх matplotlib.
from sklearn.model_selection import train_test_split  # Функция для разбиения данных
from sklearn.tree import DecisionTreeClassifier


In [ ]:
# pd.read_csv() — читает CSV-файл и создаёт DataFrame. Возвращает табличную структуру данных.
df = pd.read_csv('airline_passenger_satisfaction.csv')

# head() — возвращает первые 5 строк DataFrame (по умолчанию), удобно для быстрого просмотра структуры данных.
df.head()


In [ ]:
# drop() — удаляет указанные столбцы или строки из DataFrame

# columns — список столбцов для удаления
# inplace=True — изменяет исходный DataFrame без создания копии

df.drop(columns = ['id', 'Unnamed: 0'], inplace=True)
df.head()


In [ ]:
# dtypes — атрибут, показывающий тип данных каждого столбца
df.dtypes


## Data Cleaning


In [ ]:
df.isnull().sum()


In [ ]:
# dropna() — удаляет строки, в которых есть пропущенные значения
# inplace=True — изменяет исходный DataFrame
df.dropna(inplace=True)


In [ ]:
df.isnull().sum()


In [ ]:
df.shape


## Charts (Data Visualization)


In [ ]:
# plt.pie() — создаёт круговую диаграмму

# df['satisfaction'].value_counts() — подсчитывает количество каждого уникального значения
# labels — подписи секторов круговой диаграммы
# autopct='%1.1f%%' — формат отображения процентов

plt.pie(df['Gender'].value_counts(), labels=['Male', 'Female'], autopct='%1.1f%%')

# plt.show() — отображает график на экране
plt.show()


## Coding Data


In [ ]:
df.dtypes


In [ ]:
# LabelEncoder() — создаёт объект-кодировщик для перевода текстовых категорий в числа

label_encoder = LabelEncoder()

columns = ['Gender', 'Type of Travel','Customer Type', 'Class'] # Применяем кодирование к фиксированному списку
for col in columns:
    df[col] = label_encoder.fit_transform(df[col])




In [ ]:
df.dtypes


## Additional Charts


In [ ]:
# sns.heatmap() — создаёт тепловую карту корреляций

# df.corr() — вычисляет матрицу корреляций
# annot=True — показывает значения корреляции внутри ячеек
# fmt='.2f' — формат чисел (2 знака после запятой)
# cmap='Greens' — цветовая палитра

plt.figure(figsize=(20,10))
sns.heatmap(df.drop(columns='satisfaction').corr(), annot=True, fmt='.2f', cmap='Greens') # График корреляций
plt.show()


## Models (ML Algorithms)


In [ ]:
# df.drop(columns='satisfaction') — удаляет столбец 'satisfaction', оставляя матрицу признаков X

X=df.drop(columns = 'satisfaction')
X.head()
X.dtypes


In [ ]:
y = df['satisfaction'] # Извлекает целевую переменную (то, что хотим предсказать)
y.head()


In [ ]:
y.info()


### DecisionTree


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score, confusion_matrix  # Импорт метрик классификации
from sklearn.model_selection import train_test_split # Импорт функции для разбиения данных
df_dataset = pd.read_csv('airline_passenger_satisfaction.csv')
df_dataset.info()


In [ ]:
label_encoder = LabelEncoder()

# df_dataset.select_dtypes(include='object') — выбрать только столбцы с типом данных "object"
# .drop(columns='satisfaction') — убрать целевую переменную "satisfaction",
# так как её не нужно кодировать вместе с признаками.
# .columns — получить список имён таких текстовых столбцов.
df_dataset.select_dtypes(include = 'object').drop(columns='satisfaction').columns

for col in columns:
    df_dataset[col] = label_encoder.fit_transform(df_dataset[col])


In [ ]:
df_dataset.drop(columns = ['id', 'Unnamed: 0'], inplace=True)
df_dataset.dropna(inplace=True)
df_dataset.head()


In [ ]:
X=df_dataset.drop(columns = 'satisfaction')
X.head()


In [ ]:
y = df_dataset['satisfaction']
y.head()


In [ ]:
# ПРАВИЛЬНОЕ разбиение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y,                    # Наши данные
    test_size=0.2,          # 20% данных для теста
    random_state=42,        # Фиксируем случайность для воспроизводимости
    stratify=y              # Сохраняем пропорции классов в обеих выборках
)


In [ ]:
model = DecisionTreeClassifier() # DecisionTreeClassifier() — создаёт модель дерева решений с параметрами по умолчанию
model.fit(X_train, y_train) # fit() — обучает модель на X (признаки) и y (целевые значения)


In [ ]:
predictions = model.predict(X_test) # predict() — делает предсказания для новых данных X_test


In [ ]:
# Вычисляем все метрики качества модели
accuracy = accuracy_score(y_test, predictions)  # accuracy_score() — сравнивает истинные y_test с предсказаниями и возвращает долю совпадений
# pos_label='satisfied' -> вычислять метрику только для одного класса. Только для бинарной классификации (satisfied против neutral or dissatisfied).
# В этом режиме метрика отвечает: «Насколько хорошо модель находит именно класс 'satisfied'?»
precision = precision_score(y_test, predictions, pos_label='satisfied')  # precision_score() — вычисляет precision (из предсказанных положительных сколько действительно положительные)
recall = recall_score(y_test, predictions, pos_label='satisfied')  # recall_score() — вычисляет recall (из всех реальных положительных сколько нашли)
f1 = f1_score(y_test, predictions, pos_label='satisfied')  # f1_score() — гармоническое среднее precision и recall

# average='weighted' -> берёт оба класса и усредняет их результаты с весами по числу объектов в классе.
# average='weighted' отвечает: «В среднем по всем классам насколько хорошо работает модель»
f_beta_05 = fbeta_score(y_test, predictions, beta=0.5, average='weighted')  # fbeta_score() с beta=0.5 — больше веса на precision
f_beta_2 = fbeta_score(y_test, predictions, beta=2.0, average='weighted')  # fbeta_score() с beta=2.0 — больше веса на recall

print("📊 DECISION TREE MODEL EVALUATION RESULTS:")
print("="*50)
print(f"Accuracy:                         {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision (positive class):       {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:                           {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-Score:                         {f1:.4f} ({f1*100:.2f}%)")
print(f"F-Beta (beta=0.5, favors Precision): {f_beta_05:.4f} ({f_beta_05*100:.2f}%)")
print(f"F-Beta (beta=2.0, favors Recall):    {f_beta_2:.4f} ({f_beta_2*100:.2f}%)")

print(f"\n📝 Interpretation of the results:")
print(f"• Out of {len(y_test)} passengers, the model correctly classified {int(accuracy * len(y_test))}")
print(f"• Overall accuracy is {accuracy*100:.2f}%")


In [ ]:
# Матрица ошибок
cm = confusion_matrix(y_test, predictions)
print(f"\n🔍 CONFUSION MATRIX:")
print(f"True Negative (correctly 'dissatisfied'): {cm[0,0]}")
print(f"False Positive (wrongly 'satisfied'): {cm[0,1]}")
print(f"False Negative (wrongly 'dissatisfied'): {cm[1,0]}")
print(f"True Positive (correctly 'satisfied'): {cm[1,1]}")


### Random Forest (Homework)


In [ ]:
from sklearn.ensemble import RandomForestClassifier # RandomForestClassifier — алгоритм «случайный лес», объединяющий множество деревьев решений

model = RandomForestClassifier()
model.fit(X, y)

predictions = model.predict(X_test)
predictions

accuracy_score(y_test, predictions)


### KNeighborsClassifier (Homework)


In [ ]:
X_test.isnull().sum()


In [ ]:
from sklearn.neighbors import KNeighborsClassifier # KNeighborsClassifier — алгоритм k-ближайших соседей; n_neighbors=5 — число соседей, используемых для классификации

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X, y)

predictions = model.predict(X_test)
predictions

accuracy_score(y_test, predictions)


### Logistic Regression (Homework)


In [ ]:
from sklearn.linear_model import LogisticRegression # LogisticRegression — алгоритм логистической регрессии; max_iter=10000 — максимальное число итераций для сходимости

model = LogisticRegression(max_iter=10000)
model.fit(X, y)

predictions = model.predict(X_test)
predictions

accuracy_score(y_test, predictions)


In [ ]:
model.predict(new_df)


## Saving prediction model


In [ ]:
import joblib


In [ ]:
# joblib.dump() — сохраняет модель в файл

# model — объект модели для сохранения
# 'filename.joblib' — имя целевого файла

joblib.dump(model, 'airl_pass_satisf.joblib')


In [ ]:
trained_model = joblib.load('airl_pass_satisf.joblib') # joblib.load() — загружает сохранённую модель из файла
